# Machine Learning over Structured Data — Benchmark and Baselines

**Running example:** RelBench `rel-f1`, task `driver-position`
*For each driver who raced in the past year, predict their average finishing position over the next 60 days.*

Outline
1. Load a relational database and look at its tables, keys and time columns
2. Load a prediction task and its temporal train / val / test split
3. Baselines: global mean and **per-driver historical mean** (hands-on)
4. Flatten the database into one feature table (no leakage!)
5. Demo: TabPFN on the flat table
6. Compare results

In [6]:
%pip install -q -U "relbench>=3.0"

Note: you may need to restart the kernel to use updated packages.


## 1. The database

In [7]:
from pathlib import Path
from relbench_loader import load_oulad_relbench

import numpy as np
import pandas as pd
import relbench


raw_dir = Path(".", "output")
dataset = relbench.load_dataset(raw_dir)

db = dataset.get_db()          # rows after the test timestamp are hidden

print("val timestamp: ", dataset.val_timestamp)
print("test timestamp:", dataset.test_timestamp)
print(db)


val timestamp:  2014-02-01 00:00:00
test timestamp: 2014-10-01 00:00:00
Database(courses=22, assessments=206, vle=6364, studentInfo=32593, studentRegistration=32593, studentAssessment=122576, studentVle=7294341)


In [8]:
def print_schema(db, title="NORMALIZED DB SCHEMA", include_time=True):
    print(title)
    for i, (name, table) in enumerate(db.table_dict.items(), start=1):
        header_parts = [
            f"{i}. {name} ({len(table.df):,})",
            f"PK: {table.pkey_col}",
        ]
        if table.fkey_col_to_pkey_table:
            fk_str = ", ".join(
                f"{col} -> {target}"
                for col, target in table.fkey_col_to_pkey_table.items()
            )
            header_parts.append(f"FK: {fk_str}")
        if include_time and table.time_col is not None:
            header_parts.append(f"TIME: {table.time_col}")

        cols_str = ", ".join(
            f"{col} ({dtype})" for col, dtype in table.df.dtypes.items()
        )
        print(" | ".join(header_parts))
        print(f"   cols: {cols_str}")

print_schema(db)


NORMALIZED DB SCHEMA
1. courses (22) | PK: course_id
   cols: code_module (object), code_presentation (object), module_presentation_length (int64), raw_course_id (object), course_id (int64), course_start_date (datetime64[us]), course_end_date (datetime64[us])
2. assessments (206) | PK: id_assessment | FK: course_id -> courses
   cols: code_module (object), code_presentation (object), id_assessment (int64), assessment_type (object), date (float64), weight (float64), raw_course_id (object), course_id (int64), raw_assessment_id (int64), date_days (float64), assessment_date (datetime64[us])
3. vle (6,364) | PK: id_site | FK: course_id -> courses
   cols: id_site (int64), code_module (object), code_presentation (object), activity_type (object), week_from (float64), week_to (float64), raw_course_id (object), course_id (int64), raw_site_id (int64)
4. studentInfo (32,593) | PK: student_reg_id | FK: course_id -> courses
   cols: code_module (object), code_presentation (object), id_student (int6

In [9]:
# Every table: size, primary key, foreign keys (column -> referenced table), time column
for name, table in db.table_dict.items():
    print(f"{name:22s} rows={len(table.df):6d}  pk={table.pkey_col!s:24s} "
          f"fks={table.fkey_col_to_pkey_table}  time={table.time_col}")
    # print(table.df.head())

courses                rows=    22  pk=course_id                fks={}  time=None
assessments            rows=   206  pk=id_assessment            fks={'course_id': 'courses'}  time=None
vle                    rows=  6364  pk=id_site                  fks={'course_id': 'courses'}  time=None
studentInfo            rows= 32593  pk=student_reg_id           fks={'course_id': 'courses'}  time=None
studentRegistration    rows= 32593  pk=student_reg_id           fks={'student_reg_id': 'studentInfo', 'course_id': 'courses'}  time=None
studentAssessment      rows=122576  pk=None                     fks={'id_assessment': 'assessments', 'student_reg_id': 'studentInfo'}  time=date
studentVle             rows=7294341  pk=None                     fks={'id_site': 'vle', 'student_reg_id': 'studentInfo'}  time=date


In [11]:
courses_df = db.table_dict["courses"].df
assessments_df = db.table_dict["assessments"].df
vle_df = db.table_dict["vle"].df
studentInfo_df = db.table_dict["studentInfo"].df
studentRegistration_df = db.table_dict["studentRegistration"].df
studentAssessment_df = db.table_dict["studentAssessment"].df
studentVle_df = db.table_dict["studentVle"].df

studentAssessment_df.head()

,id_assessment,id_student,date_submitted,is_banked,score,student_reg_id,date
0,127,542386,-8,0,94.0,23459,2013-01-24
1,127,482071,-8,0,95.0,22972,2013-01-24
2,127,469743,-8,0,100.0,22932,2013-01-24
3,127,50263,-8,0,100.0,23059,2013-01-24
4,127,443696,-8,0,95.0,22905,2013-01-24


**Try it:** follow one row of `results` through its foreign keys. Which driver, which race, which circuit?

In [12]:
course = courses_df.iloc[1]
course_assessments = assessments_df.set_index("course_id").loc[course["course_id"]]
student_course_asses = studentAssessment_df[studentAssessment_df["id_assessment"].isin(course_assessments["id_assessment"])]
student_course_asses_mean = student_course_asses.groupby("id_student")["score"].mean()
print(student_course_asses_mean)

id_student
65002      67.000000
94961      74.000000
129955     81.000000
159954     66.500000
277880     67.666667
331358     84.000000
335764     86.333333
342007     75.000000
390890     69.666667
415969     59.666667
470588     86.000000
603861     63.000000
1970390    83.600000
2073016    60.333333
2139353    66.000000
2358969    68.000000
2470694    67.000000
2576122    64.000000
Name: score, dtype: float64


## 2. Prediction Task
 * Define an entity-level classification or regression task with a temporal component, where
the target is predicted from information available before a specified prediction time.
 * Define and construct the target table, with at least 10K target rows.
* Design a temporal train/validation/test split. Consider how to prevent temporal leakage.
The validation and test should contain at least 1K rows each.
* For classification tasks, make sure that the classes are relatively balanced (no class
below 10%).
* The evaluation metrics should be AUC for classification tasks, MAE and nMAE for
regression tasks (as shown in the first tutorial).

In [14]:
task = dataset.load_task("student-score") # For each driver who raced in the past year, predict their average finishing position over the next 60s days.
print("type:", task.task_type, "| entity:", task.entity_col, "| target:", task.target_col,
      "| horizon:", task.timedelta)

train_df = task.get_table("train").df
val_df = task.get_table("val").df
test_df = task.get_table("test").df      # target column is hidden

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {df.shape}  {df['timestamp'].min().date()} -> {df['timestamp'].max().date()}")
train_df.head()

type: TaskType.REGRESSION | entity: student_reg_id | target: score | horizon: 30 days 00:00:00
train (10893, 3)  2013-02-01 -> 2013-10-01
val   (5992, 3)  2014-02-01 -> 2014-02-01
test  (8935, 2)  2014-10-01 -> 2014-10-01


,timestamp,student_reg_id,score
0,2013-10-01,0,68.4
1,2013-10-01,1,72.6
2,2013-10-01,2,78.0
3,2013-10-01,3,76.6
4,2013-10-01,4,70.0


Each row is *(driverId, date, position)*: "standing at `date`, what will this driver's average finishing position be over the next 60 days?"

The split is **temporal**: all train dates < val dates < test dates.

RelBench scores regression with **NMAE** = MAE / (std of the train target). We also print plain MAE (in finishing positions), which is easier to interpret.

In [19]:
from sklearn.metrics import mean_absolute_error

val_table = task.get_table("val")
results_table = {}

def report(name, pred, field):
    """Evaluate predictions for the val rows and store the result."""
    pred = np.asarray(pred, dtype=float)
    nmae = task.evaluate(pred, val_table)["nmae"]
    mae = mean_absolute_error(val_df[field], pred)
    results_table[name] = {"val MAE": mae, "val NMAE": nmae}
    print(f"{name:25s} MAE={mae:.3f}  NMAE={nmae:.3f}")

## 3. Baseline Performance

### 3a. Global mean
1. Historical majority/mean. For each entity, collect the past records from the target table.
For classification tasks, compute the majority label; for regression, compute the mean. If
there are no historical records for the entity, use a global historical majority/mean. Use
that as the prediction for the new instance of this entity.

In [20]:
field = "score"
global_mean = train_df[field].mean()
pred_global = np.full(len(val_df), global_mean)
report("global mean", pred_global, field)

global mean               MAE=13.201  NMAE=0.939


In [41]:
train_student_df = train_df.merge(studentRegistration_df[["student_reg_id", "id_student"]], left_on="student_reg_id", right_on="student_reg_id", how="inner")
train_student_score_mean = train_student_df.groupby("id_student")[field].mean()

val_student_df = val_df.merge(studentRegistration_df[["student_reg_id", "id_student"]], right_on="student_reg_id", left_on="student_reg_id", how="inner")
pred_student_score = val_student_df["id_student"].map(train_student_score_mean).fillna(global_mean)
report("Predicted Student Score", pred_student_score, field)

train_keys = set(train_student_df["id_student"])
val_keys = set(val_student_df["id_student"])

common_keys = train_keys & val_keys
only_train_keys = train_keys - val_keys
only_val_keys = val_keys - train_keys
different_keys = train_keys ^ val_keys  # union minus intersection

print()

print(f"Unique id_student keys -> common: {len(common_keys)}, "
      f"only train: {len(only_train_keys)}, only val: {len(only_val_keys)}, "
      f"total different: {len(different_keys)}")

# Row-level check: how many val rows hit train_student_score_mean vs fallback to global_mean
in_train = val_student_df["id_student"].isin(train_student_score_mean.index)
print(f"Val rows -> matched in train: {in_train.sum()}, unseen (fallback): {(~in_train).sum()}")

Predicted Student Score   MAE=13.118  NMAE=0.933

Unique id_student keys -> common: 580, only train: 10209, only val: 5166, total different: 15375
Val rows -> matched in train: 582, unseen (fallback): 5410


## **TODO:**
2. XGBoost on the target table. Train an XGBoost model on the target table (which should
contain only the entity and timestamp), and evaluate its performance.

In [51]:
import xgboost as xgb

def extract_target_table_features(df):
    X = pd.DataFrame(index=df.index)
    X["id_student"] = df["id_student"].astype(int)
    X["year"] = pd.to_datetime(df["timestamp"]).dt.year
    X["month"] = pd.to_datetime(df["timestamp"]).dt.month
    return X

X_train_t = extract_target_table_features(train_student_df)
y_train = train_df["score"].values
X_val_t = extract_target_table_features(val_student_df)


model_b2 = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
model_b2.fit(X_train_t, y_train)
pred_b2 = model_b2.predict(X_val_t)
report("XGBoost (Target Table Only)", pred_b2, field)

# print(results_table)
for name, values in results_table.items():
    print(f"{name:25s} MAE={values["val MAE"]:.3f}  NMAE={values["val NMAE"]:.3f}")

XGBoost (Target Table Only) MAE=13.126  NMAE=0.934
global mean               MAE=13.201  NMAE=0.939
Predicted Student Score   MAE=13.118  NMAE=0.933
2. XGBoost (Target Table Only) MAE=13.126  NMAE=0.934
XGBoost (Target Table Only) MAE=13.126  NMAE=0.934


3. Deep feature synthesis. Construct join-aggregate features, both manually and
automatically (e.g., using Featuretools or your own code). Train and evaluate an
historical baseline and the target-table prediction (without additional features).

## 4. Flattening: one row per (driver, date)

TabPFN needs a single table. For each row of the task table we aggregate the driver's **past** results, meaning only races strictly before `date`. Using all of `results` would leak the future.

In [ ]:
import duckdb

FEATURE_SQL = """
SELECT
    t.driverId,
    t.date,
    COUNT(r.resultId)                                   AS n_races,
    AVG(r.positionOrder)                                AS mean_finish,
    AVG(r.grid)                                         AS mean_grid,
    SUM(r.points)                                       AS total_points,
    AVG(CASE WHEN r.date > t.date - INTERVAL 1 YEAR
             THEN r.positionOrder END)                  AS mean_finish_last_year,
    -- statusId 1 = "Finished" (same DNF definition as RelBench's driver-dnf task)
    AVG(CASE WHEN r.statusId != 1 THEN 1.0 ELSE 0 END)  AS dnf_rate,
    DATE_DIFF('day', MAX(r.date), t.date)               AS days_since_last_race,
    DATE_DIFF('day', ANY_VALUE(d.dob), t.date) / 365.25 AS age
FROM task_rows t
JOIN drivers d ON d.driverId = t.driverId
LEFT JOIN results r
       ON r.driverId = t.driverId
      AND r.date < t.date            -- only the past!
GROUP BY t.driverId, t.date
"""

def make_features(task_df):
    con = duckdb.connect()
    con.register("task_rows", task_df[["driverId", "date"]])
    con.register("drivers", drivers)
    con.register("results", results)
    feats = con.sql(FEATURE_SQL).df()
    con.close()
    # keep the task table's row order
    return task_df[["driverId", "date"]].merge(feats, on=["driverId", "date"], how="left")

X_train_full = make_features(train_df)
X_val_full = make_features(val_df)
FEATURES = [c for c in X_train_full.columns if c not in ("driverId", "date")]
X_train_full.head()

**Checkpoint:** why `r.date < t.date` and not just "all results of the driver"?
What would happen to the validation score if we removed that condition?

## 5. Demo: TabPFN

TabPFN is a transformer **pre-trained on millions of synthetic tables**. `fit` stores the training rows; `predict` runs one forward pass with the training rows as context ("in-context learning").

* GPU recommended. On CPU, keep ≤ ~5,000 training rows.
* The model weights download on first use and may require accepting a license on Hugging Face. **Do this before class.**

In [ ]:
!pip install -q tabpfn

In [ ]:
import torch
from tabpfn import TabPFNRegressor

MAX_TRAIN = 10_000 if torch.cuda.is_available() else 5_000
train_sub = (X_train_full.assign(y=train_df["position"].to_numpy())
             .sort_values("date").tail(MAX_TRAIN))      # most recent rows
print("training rows used:", len(train_sub), "| device:", "cuda" if torch.cuda.is_available() else "cpu")

reg = TabPFNRegressor()
reg.fit(train_sub[FEATURES], train_sub["y"])
pred_tabpfn = reg.predict(X_val_full[FEATURES])
report("TabPFN (flat features)", pred_tabpfn)

**Ablation:** is the gain from the model, or from the features?
Give TabPFN only a single feature, `mean_finish` (≈ the historical mean), and compare.

In [ ]:
reg_1f = TabPFNRegressor()
reg_1f.fit(train_sub[["mean_finish"]], train_sub["y"])
report("TabPFN (mean_finish only)", reg_1f.predict(X_val_full[["mean_finish"]]))

## 6. Compare

In [ ]:
pd.DataFrame(results_table).T.sort_values("val MAE")

Discussion
1. Does TabPFN beat the per-driver mean? By how much?
2. Is the improvement due to the model or to the features we gave it?
3. Which other tables (`qualifying`, `standings`, `constructors`, ...) could add signal, and how would you keep them leak-free?